    # Concise Temperature Forecasting Project

Predict temperature six hours ahead from 24 hourly observations. Run each step and commit after successful execution.

Data Source: [MPI-BGC Beutenberg](https://weather.bgc-jena.mpg.de/weather_data.html), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

# 1. Environment and Setup
Fix model settings and create result folders.


In [1]:
import os
import tempfile
from pathlib import Path

# Keep Windows temporary files in the project.
Path(".runtime").mkdir(exist_ok=True)
tempfile.tempdir = str(Path(".runtime").resolve())
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_NUM_INTRAOP_THREADS"] = "4"
os.environ["TF_NUM_INTEROP_THREADS"] = "2"

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import truststore
truststore.inject_into_ssl()
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

SEED, LOOKBACK, HORIZON = 504, 24, 6
FEATURES = ["T (degC)", "p (mbar)", "rh (%)", "wv (m/s)"]
for folder in ["data", "outputs", "models", "figures"]:
    Path(folder).mkdir(exist_ok=True)
tf.config.experimental.enable_op_determinism()
CONFIG = {"seed": SEED, "lookback": LOOKBACK, "horizon": HORIZON,
          "features": FEATURES, "learning_rate": 0.001, "batch_size": 64,
          "max_epochs": 30, "patience": 5}
Path("outputs/config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Runtime versions:", keras.__version__, tf.__version__)

Runtime versions: 3.15.1 2.21.0


# 2. Download and clean
Source: MPI-BGC Beutenberg, CC BY 4.0. Check duplicate conflicts and mask invalid values.

In [2]:
from urllib.request import urlretrieve
from zipfile import ZipFile

names = [f"mpi_roof_{year}{half}" for year in range(2018, 2024) for half in "ab"]
names += ["mpi_roof_2024", "mpi_roof_2025"]
parts = []
for name in names:
    path = Path("data") / f"{name}.zip"
    if not path.exists():
        urlretrieve(f"https://www.bgc-jena.mpg.de/wetter/data/{name}.zip", path)
    with ZipFile(path) as archive:
        with archive.open(f"{name}.csv") as file:
            parts.append(pd.read_csv(file, encoding="latin-1"))

data = pd.concat(parts, ignore_index=True)
data["Date Time"] = pd.to_datetime(data["Date Time"], format="%d.%m.%Y %H:%M:%S")
# Resolve conflicting measurements before deduplication.
repeated = data[data["Date Time"].duplicated(keep=False)]
measurements = data.columns.drop("Date Time")
assert not repeated.groupby("Date Time")[measurements].nunique(dropna=False).gt(1).any().any()
observed = (data.sort_values("Date Time").drop_duplicates("Date Time")
            .set_index("Date Time")[FEATURES].copy())
observed = observed.replace([-9999, np.inf, -np.inf], np.nan)
observed.loc[observed["p (mbar)"] <= 0, "p (mbar)"] = np.nan
observed.loc[~observed["rh (%)"].between(0, 100), "rh (%)"] = np.nan
observed.loc[observed["wv (m/s)"] < 0, "wv (m/s)"] = np.nan
print("Raw/deduplicated rows:", len(data), len(observed))
display(observed.isna().sum().to_frame("Missing count"))

Raw/deduplicated rows: 420782 420631


,Missing count
T (degC),0
p (mbar),0
rh (%),0
wv (m/s),1
